In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

In [2]:
data_clean = "goodreads_onestar_clean.csv"   # cleaned data, one-star reviews only
TEXT_COL = "review_text"   # name of the column with the review text
MIN_WORDS = 10             # skip reviews shorter than this many words
SEED = 42 # random seed for reproducibility

HIDE = ["alice", "wonderland", "carroll", "silverstein", "peter"] # common titles and authors to hide

In [3]:
one_star = pd.read_csv(data_clean) # turning into a DataFrame
one_star = one_star.dropna(subset=[TEXT_COL]) # dropping reviews with no text
one_star = one_star[one_star[TEXT_COL].str.split().str.len() >= MIN_WORDS] # splitting review text into list of words, counting them, and keeping only reviews with at least MIN_WORDS words

print(f"1-star reviews: {len(one_star)}") # printing the number of 1-star reviews after filtering

1-star reviews: 8801


### Flagging reviews about "appropriateness"

In [4]:
def compare_groups(texts_a, texts_b):  # scoring words and phrases that are distinctive to one group vs another
    texts_a = list(texts_a) # a positive score = more typical of group A, negative score = more typical of group B
    texts_b = list(texts_b)

    counter = CountVectorizer( # using CountVectorizer to count words and phrases in the two groups
        ngram_range=(1, 2), # defining upper and lower limit of words to be extracted
        min_df=15,    # words/phrases must appear in 15+ reviews
        token_pattern=r"(?u)\b\w[\w']*\b",   # keeps negations and other words with apostrophes like "don't" 
    )
    counts = counter.fit_transform(texts_a + texts_b) # makes table of counts of each word/phrase in each review
        
    a_review_number = len(texts_a) # how many reviews in group A and group B
    b_review_number = len(texts_b)
    count_a = np.asarray(counts[:a_review_number].sum(axis=0)).ravel() # selects group A reviews, sums the counts of each term across all reviews, and flattens into list of numbers
    count_b = np.asarray(counts[a_review_number:].sum(axis=0)).ravel() # selects group B reviews, sums the counts of each term across all reviews, and flattens into list of numbers

    baseline = count_a + count_b # how many times each term appears in all reviews (group A + group B)
    baseline_total = baseline.sum() # sums across all terms to get the total number of term appearances in all reviews

    odds_a = np.log((count_a + baseline) / (count_a.sum() + baseline_total - count_a - baseline)) # computes the log-odds of each term appearing in group A vs group B, with a baseline to avoid division by zero
    odds_b = np.log((count_b + baseline) / (count_b.sum() + baseline_total - count_b - baseline))
    difference = odds_a - odds_b

    # Turn the difference into a score that accounts for how much data there is
    uncertainty = np.sqrt(1 / (count_a + baseline) + 1 / (count_b + baseline)) # this calculates uncertainty for each term based on its counts in both groups and the baseline 
    score = difference / uncertainty # dividing to get a z-score-like measure

    results = pd.DataFrame({ # putting everything into a DataFrame 
        "term": counter.get_feature_names_out(),
        "score": score,
        "count_A": count_a,
        "count_B": count_b,
    })

        
    for name in HIDE: # hiding common titles and authors from the results
        results = results[~results["term"].str.contains(r"\b" + name + r"\b")]

    return results.sort_values("score", ascending=False).reset_index(drop=True) # sorting the results by score in descending order and resetting the index


In [5]:
# These rules only record which phrase caused a review to enter the candidate set.
# They are NOT hand-label categories and do not describe what is inappropriate.
SCREENING_RULE_PATTERNS = {
    "explicit_phrase": (
        r"\b(?:age[- ]?)?inappropriate\w*\b"
        r"|\b(?:not|isn't|wasn't|aren't|weren't)\s+(?:\w+\s+){0,2}(?:age[- ]?)?appropriate\b"
        r"|\bnot\s+suitable\b|\bunsuitable\b"
    ),
    "evaluative_term": r"\b(?:offensive|vulgar|crude|perverse|immoral|sinful)\b",
    "intensity_phrase": r"\btoo\s+(?:scary|violent|graphic|dark|mature|disturbing|sexual|adult|intense|creepy)\b",
    "age_fit_phrase": r"\btoo\s+(?:old|young|mature|advanced)\s+for\b",
    "audience_phrase": (
        r"\bnot\s+(?:for|meant\s+for|intended\s+for)\s+(?:\w+\s+){0,2}(?:kids|children|young)\b"
        r"|\bnot\s+(?:very\s+)?(?:child|kid|family)[- ]friendly\b"
    ),
}

# Use a non-capturing pattern for screening and named groups for match details.
TRIGGER_PATTERN = "|".join(f"(?:{pattern})" for pattern in SCREENING_RULE_PATTERNS.values())
LABELED_TRIGGER_PATTERN = "|".join(
    f"(?P<{name}>{pattern})" for name, pattern in SCREENING_RULE_PATTERNS.items()
)
WINDOW = 12

In [6]:
has_trigger = one_star[TEXT_COL].str.contains(TRIGGER_PATTERN, case=False, regex=True) # checking which reviews contain the trigger pattern, ignoring case and using regex
matched = one_star[has_trigger]
others = one_star[~has_trigger]
print(f"{len(matched):,} of {len(one_star):,} 1-star reviews contain an " # printing the number and percentage of 1-star reviews that contain an inappropriateness complaint
      f"inappropriateness complaint ({len(matched) / len(one_star):.1%})")

224 of 8,801 1-star reviews contain an inappropriateness complaint (2.5%)


In [7]:
def first_match_details(text, width=WINDOW):
    m = re.search(LABELED_TRIGGER_PATTERN, text, flags=re.I)
    before = " ".join(text[:m.start()].split()[-width:])
    after = " ".join(text[m.end():].split()[:width])
    return pd.Series({
        "matched_trigger": m.group(0),
        "screening_rule": m.lastgroup,
        "snippet": f"{before} [[{m.group(0)}]] {after}",
    })


def all_triggers(text):
    # Preserve order while removing duplicate matched phrases.
    matches = (m.group(0) for m in re.finditer(LABELED_TRIGGER_PATTERN, text, flags=re.I))
    return " | ".join(dict.fromkeys(matches))


matched = matched.copy()
matched[["matched_trigger", "screening_rule", "snippet"]] = matched[TEXT_COL].apply(first_match_details)
matched["all_triggers"] = matched[TEXT_COL].apply(all_triggers)

pd.set_option("display.max_colwidth", None) # letting the entire snippet show up in the output, instead of being truncated
print("\n25 random examples (read these first):")
print(matched[["book_id", "screening_rule", "snippet"]].sample(min(25, len(matched)), # printing 25 random examples of the matched reviews, showing the book_id and the snippet, and using a random seed for reproducibility
                                             random_state=SEED).to_string())


25 random examples (read these first):
       book_id    screening_rule                                                                                                                                                            snippet
105     237564   audience_phrase                             This book is a choking hazard and contains small parts. It is [[not for children]] under 3 years." That said, I read it to my 16 month
4087  16113648   evaluative_term                                                                                          Yup. Every bit as [[offensive]] as all the reviews and articles attested.
5639    389627   evaluative_term                    failures is because at an early age these days they're taught that [[crude]] humor is funny, that apathy and laziness is cute and that bullying
6647     65119  intensity_phrase                     I'm glad someone enjoys these books really. But the material is way [[too violent]] for young kids and yet the writing styl

In [8]:
distinct = compare_groups(matched[TEXT_COL], others[TEXT_COL]) # running the compare_groups function on the matched and others DataFrames to get the most distinctive words in the matched reviews
distinct = distinct[~distinct["term"].str.contains("appropriat|suitab|offen")] # removing the trigger words from the results, so that the most distinctive words are not just the trigger words themselves
distinct.to_csv("words_inappropriate_vs_other_1star.csv", index=False)
print("\nMost distinctive words in reviews that call something inappropriate:")
print(distinct.head(40).round(2).to_string())


Most distinctive words in reviews that call something inappropriate:
             term  score  count_A  count_B
5         old for   4.09       18        5
6         too old   3.39       18       15
7            rose   3.36       23       30
9            dark   2.98       45      159
10             oz   2.87       25       54
11            god   2.82       51      217
12  www goodreads   2.79       15       18
13  goodreads com   2.74       15       19
14       the dark   2.54       21       49
15      https www   2.45       14       22
16         my kid   2.45       17       34
17          bible   2.39       23       66
18          of oz   2.19       14       29
19          https   2.16       14       30
20           rain   2.04       13       29
21       children   2.02      158     1651
22          god's   1.97        9       14
23      preschool   1.97       11       22
24            www   1.92       19       68
25   for children   1.85       36      212
26           2014   1.83   

#### CSV containing all candidate reviews and blank HAND LABEL columns

`screening_rule` and `matched_trigger` only document why a review was retrieved. They must not be used as the final hand-label categories. The substantive category belongs in the blank `aspect` column after reading the full review. All reviews that match the screening rules are exported; the code does not impose a target sample size.

In [9]:
# Export every candidate rather than dropping reviews through random sampling.
# The original row index is retained as a stable ID when no review_id is available.
matched = matched.rename_axis("source_row_id").reset_index()
id_columns = [column for column in ["review_id", "source_row_id", "book_id"] if column in matched.columns]
screening_columns = ["matched_trigger", "screening_rule", "all_triggers", "snippet"]
to_code = matched[id_columns + [TEXT_COL] + screening_columns].copy()

# These are the only labeling fields. Screening metadata above is not a label.
to_code["is_inappropriate_complaint"] = ""
to_code["aspect"] = ""
to_code["notes"] = ""

to_code.to_csv("inappropriate_to_code.csv", index=False, encoding="utf-8-sig")
print("\nSaved inappropriate_to_code.csv with", len(to_code), "reviews to hand label")


Saved inappropriate_to_code.csv with 224 reviews to hand label


In [10]:
hand_labeled = pd.read_csv("inappropriate_to_code.csv", encoding="utf-8-sig")

hand_labeled.head()

,review_id,source_row_id,book_id,review_text,matched_trigger,screening_rule,all_triggers,snippet,is_inappropriate_complaint,aspect,notes
0,ee39ef35634d9dbe72d559db4aded36e,11,156970,"I guess I am too old for this book. Even though I did like the fact it had a female protagonist, it still was a little too ""kiddy"" for me. Damn it I should have read it earlier in my life, maybe I would have given it a 5 stars then.",too old for,age_fit_phrase,too old for,I guess I am [[too old for]] this book. Even though I did like the fact it had a,NaN,NaN,NaN
1,28acc4d2ed1b1798ad94066cdbfe3d45,33,3350840,"This series is awful. Listed on CBD for ages 5-10, and on Amazon as 4 and up, this designation is completely off, even arbitrary. I would recommend these books, maybe, for preschool. For some reason, they decided to use rhyming verse, a literary style that greatly appeals to the preschool set. But in that format, the serious topics of missions, paganism, and the confrontation between light and darkness are addressed. \n This is everything I hate about cheery ""christian"" literature. A sing-song, happy-happy-clap-clap approach to the very real war of the ages is infuriatingly inappropriate in preparing our children to stand for the LORD. If this kind of God will never let anything uncomfortable happen to you sweet child theology is what you are looking for, Meloche's work will suit you fine. But if you are looking to raise up godly children who are equipped to know, love and serve the amazing, awesome, Creative God, do yourself a favor: Don't buy any books in this series. \n Excellent alternatives, which I have reviewed on Goodreads are: \n Christian Biographies for Young Readers (Series), Simonetta Carr \n https://www.goodreads.com/series/8395... \n Trial & Tribulation (A single book overview of Church History), Hannula \n https://www.goodreads.com/review/show... \n History Lives Series by Brandon & Mindy Withrow \n https://www.goodreads.com/review/show...",inappropriate,explicit_phrase,inappropriate,happy-happy-clap-clap approach to the very real war of the ages is infuriatingly [[inappropriate]] in preparing our children to stand for the LORD. If this kind,NaN,NaN,NaN
2,a829e1ebbb4b6d7655b219b401c58d95,34,430332,"Once again, the author brings up issues to complex for the rhyming scheme. From pages, oops, sorry there are no page numbers. \n ""As he untied the string that held \n the book that he had hidden, \n he knew he could be punished, for \n his father had forbidden \n all books containing science. He \n and many people thought \n that if you were Christian, science \n should never be taught. \n For many living in the \n nineteenth century believed \n it challenged faith in God to question \n how things were conceived, \n like plants and animals, \n and even people too. \n But David longed to study and \n learn how all life grew."" \n Furthermore, her animosity between Christians and science misrepresents David Livingstone. In other reading, it appears to me that Mr. Livingstone's issue with science was not that it shouldn't be studied at all, but that Biblical study should take precedence in the education of his children. I would say this is also true of Christians today. It's not that science is scary, evil, or lacks value, but that the Bible should be preeminent. Meanwhile, young David Livingstone's issue was that his own scientific curiosity lead him to rebel against his father, which he knew God did not view favorably. \n For a superior treatment of Livingstone, see Sam Wellman's, David Livingstone, from the Young Readers Christian Library series: \n https://www.goodreads.com/review/show... \n This series is awful. Listed on CBD for ages 5-10, and on Amazon as 4 and up, this designation is completely off, even arbitrary. I would recommend these books, maybe, for preschool. For some reason, they decided to use rhyming verse, a literary style that greatly appeals to the preschool set. But in that format, the serious topics of miss

### Looking at NOT-flagged reviews (reviews not about "appropriateness")

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

print("\nWords most typical of the NOT-flagged reviews:")
print(distinct.sort_values("score").head(40).round(2).to_string(index=False)) # printing the 40 words with the lowest score, which are the most typical of the NOT-flagged reviews



Words most typical of the NOT-flagged reviews:
        term  score  count_A  count_B
           i  -1.12      989    22475
      boring  -0.84       18      759
    couldn't  -0.83       11      559
 interesting  -0.82       10      523
  half stars  -0.80        1      229
        baby  -0.80        4      328
      a half  -0.79        2      261
     strange  -0.79        1      221
         put  -0.76        7      404
     one and  -0.76        4      311
          it  -0.75      657    14583
        half  -0.75        9      455
       sense  -0.74        9      450
        been  -0.74       38     1198
        it's  -0.73      103     2699
        nice  -0.71        7      379
    annoying  -0.69        4      280
        done  -0.69        6      338
   it wasn't  -0.67        1      170
   finish it  -0.67        0      131
  i couldn't  -0.66        4      264
     friends  -0.66       10      436
      follow  -0.66        1      165
        fish  -0.66        1      165
  

In [12]:
print("\n15 random NOT-flagged reviews (first 300 characters):")
for text in others[TEXT_COL].sample(15, random_state=SEED): # a loop taking a random sample of 15 NOT-flagged reviews
    print("-", text[:300].replace("\n", " "), "\n")


15 random NOT-flagged reviews (first 300 characters):
- I typically love anything touched by Jon scieszka. I suppose I need to be an elementary-aged boy to get it? I don't get it at all. The book is totally annoying. 

- This is the lowest rating I've ever given to a book that I liked, but the problematic section with the slaves ruins the rest of the beautiful experience. It's gut-wrenching to behold. :( 

- Why in the world didn't they put a shirt on the little girl! Blech! Children in books should be clothed! 

- WHAT the what was this? Why did I waste my time???!!!   Could I please get those 5 minutes back? And the brain cells it killed, I need those, too, please.   Oy vey. 

- Yuck. Just yuck. This third book in the "Very first..." series which has previously covered Easter and Christmas doesn't make any improvement. I much prefer Ben Avery's work on the Twelve Apostles, the Torchlighter DVD on Perpetua, and I hear Simonetta Carr has a book on Irenaeus coming out as well. 

- I th

#### Topic Model of NOT-flagged reviews

In [13]:
NUMBER_OF_TOPICS = 8
tfidf = TfidfVectorizer(stop_words="english", min_df=10, max_df=0.5) # using TF-IDF to assign weights to words in the reviews
X = tfidf.fit_transform(others[TEXT_COL]) # transforming the NOT-flagged reviews into a matrix
nmf = NMF(n_components=NUMBER_OF_TOPICS, random_state=SEED, init="nndsvda", max_iter=400) # using NMF to find topics in the NOT-flagged reviews, with 8 topics, a random seed for reproducibility, and a maximum of 400 iterations
doc_topics = nmf.fit_transform(X) # transforming the matrix of NOT-flagged reviews into a matrix of topic weights for each review
terms = tfidf.get_feature_names_out() # getting the feature names (words) from the TF-IDF vectorizer

others = others.copy()
others["topic"] = doc_topics.argmax(axis=1) # assigning each review to the topic with the highest weight
for i, weights in enumerate(nmf.components_): # a loop that goes through each topic and prints the top 10 words for that topic, along with the number of reviews assigned to that topic
    top_words = [terms[j] for j in weights.argsort()[::-1][:10]]
    n_reviews = (others["topic"] == i).sum()
    print(f"\nTopic {i} ({n_reviews:,} reviews): {', '.join(top_words)}")


Topic 0 (2,364 reviews): children, little, child, way, good, time, people, old, things, boy

Topic 1 (968 reviews): didn, like, really, did, felt, care, enjoy, wanted, characters, feel

Topic 2 (972 reviews): read, hated, school, ve, grade, time, remember, reading, aloud, did

Topic 3 (1,170 reviews): story, illustrations, text, cute, nice, pictures, good, words, liked, stars

Topic 4 (1,003 reviews): just, couldn, finish, maybe, really, bad, wasn, funny, sorry, got

Topic 5 (423 reviews): boring, finish, plot, thought, really, long, ugh, good, finished, dull

Topic 6 (856 reviews): don, know, think, kids, like, understand, kid, want, maybe, remember

Topic 7 (821 reviews): books, series, reading, better, love, picture, ve, enjoy, good, kids


## LLM Labeling. --By Melody 11-23version

Basic Logic:
1. use Gemini API key to first label the random 50 reviews from the whole inapproproate data set and make some polishment for the labels' instruction/rule/definition 

2. and then apply the latest version of the labels' instruction/rule/definition to the whole inappropriate data set.

Same workflow as HW5, with one difference: **a review can have several labels**.

| Step | What happens |
|---|---|
| 0 | Settings, Gemini connection, labels and definitions |
| 1 | Sample 50 reviews from the team's latest CSV and export them for hand labeling |
| 2 | Gemini labels the same 50 with the v1 instruction |
| 3 | Accuracy, per-label precision/recall, and the reviews where Gemini and the hand labels differ |
| 4 | The group polishes the definitions (v2) |
| 5 | Label **all** reviews with v2 and save the final CSV with the LLM labels |
| 6 | Use the v2 for in depth data visualization for the next step |


This version is written for the team's latest file, `inappropriate_reviews_llm_labels.csv`:

| Column | What it is | Sent to Gemini? |
|---|---|---|
| `review_id` | unique id | no (only used to match answers) |
| `book_id` | book id | no |
| `review_text` | the review | **yes, this is the only thing Gemini sees** |
| `labels` | the team's hand labels | **no**, used only to score Gemini afterwards |
| `notes` | the team's notes | **no** |
| `llm_labels` | **filled in by this notebook** | – |
| `llm_reasoning` | **filled in by this notebook** | – |


### 0-1 Settings and connection

In [14]:
import ast, json, os, time
from pathlib import Path

import numpy as np
import pandas as pd

# ====== CHANGE THESE ======
DATA_PATH = Path("data/inappropriate_reviews_llm_labels.csv")   # the team's latest CSV
ID_COL = "review_id"
TEXT_COL = "review_text"      # the ONLY column sent to Gemini
HAND_COL = "labels"           # hand labels: used for scoring only, never sent to Gemini
OUT_DIR = Path("llm_output")  # everything this notebook creates goes here
SAMPLE_SIZE = 50
SEED = 211
SLEEP_SECONDS = 6             # wait between calls to stay under the free tier's limit per minute
# ==========================

OUT_DIR.mkdir(exist_ok=True)

In [17]:
# Connect to Gemini (same as HW5). The key is read from .env and never printed.
import logging

from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import errors, types

load_dotenv(find_dotenv(usecwd=True))
key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"  # use the replacement model if bCourses announces one

logging.getLogger("google_genai.types").setLevel(logging.ERROR)
print("Connected to Gemini.")

Connected to Gemini.


In [18]:
import sys
print(sys.executable)

/Users/sofiaabbas/Desktop/GitHub/Goodreads-Team/.venv/bin/python


### 0-2 Labels and definitions

The rules come from the team's Goodreads Project Outline. **All definitions live in `LABEL_RULES` and `EXTRA_RULES`**, and the instruction text is built from them

***a problem:The hand labels in the CSV also contain `preachy` and `sexuality`, which are not in the outline's list of 16 labels, so Gemini does not know them.


In [31]:
LABEL_RULES = {
    "age-appropriateness": "The review says the book is not appropriate for children. Do NOT use it when the reviewer only says they themselves are too old for the book.",
    "religion": "Religion or spirituality is part of the complaint (use only when it is part of the complaint).",
    "racism": "The complaint is about race, racism (including so-called reverse racism), racist or ethnic stereotyping, or xenophobia.",
    "body-shaming": "The complaint is about body-shaming, body image, or fatphobia.",
    "bad-influence": "The book has a bad influence on kids, or teaches kids wrong or immoral behaviors and lessons.",
    "scary/dark": "The book is too scary, dark, disturbing, creepy, or described with similar words.",
    "illustrations": "The book's illustrations are part of the complaint (use only when illustrations are part of the complaint).",
    "abuse": "The complaint is about the book's portrayal of physical, psychological, or sexual abuse (the review explicitly mentions abuse).",
    "ableism": "The complaint is about the treatment of disabled people, including physical disabilities and developmental disabilities such as autism.",
    "non-factual": "The book's content gives incorrect information.",
    "sexism": "The complaint is about sexism, misogyny, patriarchy, or a negative or stereotypical portrayal of women and girls.",
    "classism": "The complaint is about classism, wealth superiority, or elitism.",
    "violence": "The complaint is about violence or any violent or harmful scenario.",
    "witchcraft": "Witchcraft is part of the complaint (use only when witchcraft is part of the complaint).",
    "crude-language": "The complaint is about crude, suggestive, sexual, or vulgar language or illustrations.",
    "drugs/alcohol": "The complaint is about the portrayal of drugs and/or alcohol in the book.",
    "preachy": "The complaint mentions the book being preachy.",
    "sexuality": "The complaint mentions sexuality, sexual orientation, or related words such as queer, gay, lesbian, homophobic, and homosexual.",
}
LABELS = list(LABEL_RULES)

EXTRA_RULES = [
    "Label by the topic of the complaint. Give EVERY label whose rule is met. Give none if no rule is met.",
    "Some complaints are inappropriate regardless of age (for example racist content). Do not add age-appropriateness for those unless the reviewer also says the book is not appropriate for children.",
    "bad-influence and drugs/alcohol are always about a young audience: whenever you give either one, also give age-appropriateness.",
    "Base the labels only on what the reviewer actually says. Do not infer a complaint that is not in the text.",
]


def build_system(rules, extra):
    lines = [
        "You are labeling one-star Goodreads reviews of children's books that were flagged as possibly complaining about inappropriateness.",
        "Decide which of the labels below apply to the review. A review can have several labels, one label, or none.",
        "",
        "Labels:",
    ]
    lines += [f"- {name}: {rule}" for name, rule in rules.items()]
    lines += ["", "Rules:"] + [f"- {r}" for r in extra]
    lines += ["", "Return llm_labels (a list of label names copied exactly as written above; an empty list if none apply) and llm_reasoning (one short sentence)."]
    return "\n".join(lines)


SYSTEM_V1 = build_system(LABEL_RULES, EXTRA_RULES)


def build_schema(labels):
    """The reply format Gemini must follow."""
    return {
        "type": "object",
        "properties": {
            "llm_labels": {"type": "array", "items": {"type": "string", "enum": list(labels)}},
            "llm_reasoning": {"type": "string"},
        },
        "required": ["llm_labels", "llm_reasoning"],
    }


SCHEMA = build_schema(LABELS)   # if you add labels later, rebuild this from the new LABELS
print(SYSTEM_V1)

You are labeling one-star Goodreads reviews of children's books that were flagged as possibly complaining about inappropriateness.
Decide which of the labels below apply to the review. A review can have several labels, one label, or none.

Labels:
- age-appropriateness: The review says the book is not appropriate for children. Do NOT use it when the reviewer only says they themselves are too old for the book.
- religion: Religion or spirituality is part of the complaint (use only when it is part of the complaint).
- racism: The complaint is about race, racism (including so-called reverse racism), racist or ethnic stereotyping, or xenophobia.
- body-shaming: The complaint is about body-shaming, body image, or fatphobia.
- bad-influence: The book has a bad influence on kids, or teaches kids wrong or immoral behaviors and lessons.
- scary/dark: The book is too scary, dark, disturbing, creepy, or described with similar words.
- illustrations: The book's illustrations are part of the compla

#### 1. Load the table and sample 50 reviews

In [20]:
# keep_default_na=False keeps empty cells as "" instead of NaN, so the table is written back exactly as it was read.
DATA_PATH = Path("inappropriate_reviews_llm_labels.csv")

data = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
ORIGINAL_COLUMNS = list(data.columns)

for col in (ID_COL, TEXT_COL, HAND_COL, "llm_labels", "llm_reasoning"):
    assert col in data.columns, f"Column {col!r} is missing. The columns are: {ORIGINAL_COLUMNS}"
assert data[ID_COL].is_unique, "review_id is not unique"
assert (data[TEXT_COL].str.strip() != "").all(), "Some reviews have no text"
print(data.shape, ORIGINAL_COLUMNS)

# What Gemini is allowed to see: ONLY the id and the review text. labels and notes are not in this table at all.
to_label = data[[ID_COL, TEXT_COL]].rename(columns={ID_COL: "id", TEXT_COL: "text"})

sample = to_label.sample(n=SAMPLE_SIZE, random_state=SEED)
print(len(to_label), "reviews in total;", len(sample), "in the sample")

(166, 7) ['review_id', 'book_id', 'review_text', 'labels', 'notes', 'llm_labels', 'llm_reasoning']
166 reviews in total; 50 in the sample


In [21]:
# The hand labels ("['age-appropriateness', 'racism']" -> a set of label names). Used ONLY to score Gemini.
def parse_hand(cell):
    try:
        items = ast.literal_eval(cell)
        assert isinstance(items, (list, tuple))
    except (ValueError, SyntaxError, AssertionError):
        items = str(cell).replace("[", "").replace("]", "").replace("'", "").split(",")
    return {str(x).strip().lower() for x in items if str(x).strip()}


hand_sets = {rid: parse_hand(cell) for rid, cell in zip(data[ID_COL], data[HAND_COL])}

# Hand labels that Gemini has no definition for (it can never predict them):
extra_hand_labels = sorted({lab for s in hand_sets.values() for lab in s} - set(LABELS))
print("Hand labels not in LABELS:", extra_hand_labels or "none")

Hand labels not in LABELS: none


#### 2. Gemini labels the 50 with the v1 instruction

In [22]:
# Supplied (same as HW5, changed to many labels per review).
# Errors worth trying again, because they go away on their own: 429 = too many calls in one minute, 503 = server busy.
RETRY_CODES = {429, 503}


def ask_gemini(prompt, system, schema, max_tries=4):
    """Send one prompt with this system instruction and schema. Return the labels and the reasoning. Try again after a temporary error."""
    wait = 10  # seconds to wait before the first retry
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    response_mime_type="application/json",
                    response_schema=schema,
                ),
            )
            time.sleep(SLEEP_SECONDS)
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in RETRY_CODES or attempt == max_tries:
                raise  # stop and show the error
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2  # wait longer each time: 10, 20, 40 seconds


def label_texts(texts, system, schema, path):
    """Ask Gemini about every review in texts (columns id and text only). Save each answer to path right away.
    Reviews that are already in the file are skipped, so you can run it again after an interruption."""
    done = set(pd.read_csv(path, dtype={"id": str})["id"]) if path.exists() else set()
    for row in texts.itertuples():
        if row.id in done:
            continue
        reply = ask_gemini(f"Review: {row.text}", system, schema)
        labels = list(dict.fromkeys(lab for lab in reply.get("llm_labels", []) if lab in schema_labels(schema)))  # valid names, no repeats
        out = pd.DataFrame([{
            "id": row.id,
            "llm_labels": str(labels),                      # same format as the hand labels: "['a', 'b']", or "[]"
            "llm_reasoning": reply.get("llm_reasoning", ""),
            "model": GEMINI_MODEL,
        }])
        out.to_csv(path, mode="a", header=not path.exists(), index=False)
        done.add(row.id)
        print(len(done), "of", len(texts), "done")
    results = pd.read_csv(path, dtype=str, keep_default_na=False)
    return results[results["id"].isin(texts["id"])]


def schema_labels(schema):
    return schema["properties"]["llm_labels"]["items"]["enum"]

In [24]:
# Label the 50 with the v1 instruction (50 calls, a few minutes). If it stops, run this cell again: it skips the reviews already done.
v1_results = label_texts(sample, SYSTEM_V1, SCHEMA, OUT_DIR / "llm_v1_sample50_raw.csv")
len(v1_results)   # Check: 50

Error 503. Waiting 10 seconds, then trying again.
18 of 50 done
Error 503. Waiting 10 seconds, then trying again.
19 of 50 done
Error 503. Waiting 10 seconds, then trying again.
Error 503. Waiting 20 seconds, then trying again.
Error 503. Waiting 40 seconds, then trying again.
20 of 50 done
Error 503. Waiting 10 seconds, then trying again.
Error 503. Waiting 20 seconds, then trying again.
Error 503. Waiting 40 seconds, then trying again.
21 of 50 done
22 of 50 done
23 of 50 done
24 of 50 done
25 of 50 done
Error 503. Waiting 10 seconds, then trying again.
26 of 50 done
27 of 50 done
Error 503. Waiting 10 seconds, then trying again.
28 of 50 done
Error 503. Waiting 10 seconds, then trying again.
29 of 50 done
30 of 50 done
Error 503. Waiting 10 seconds, then trying again.
Error 503. Waiting 20 seconds, then trying again.
Error 503. Waiting 40 seconds, then trying again.
31 of 50 done
32 of 50 done
33 of 50 done
Error 503. Waiting 10 seconds, then trying again.
34 of 50 done
35 of 50 don

50

In [25]:
def fill_columns(table, results, only_ids=None):
    """Return the team's table (same columns, same order) with llm_labels and llm_reasoning filled from results."""
    out = table[ORIGINAL_COLUMNS].copy()
    out["llm_labels"] = out[ID_COL].map(dict(zip(results["id"], results["llm_labels"]))).fillna("")
    out["llm_reasoning"] = out[ID_COL].map(dict(zip(results["id"], results["llm_reasoning"]))).fillna("")
    if only_ids is not None:
        out = out[out[ID_COL].isin(only_ids)]
    return out


v1_filled = fill_columns(data, v1_results, only_ids=sample["id"])
v1_filled.to_csv(OUT_DIR / "inappropriate_reviews_llm_v1_sample50.csv", index=False)
print("Saved the 50 sampled reviews with llm_labels and llm_reasoning filled in:", v1_filled.shape)
v1_filled[[ID_COL, HAND_COL, "llm_labels", "llm_reasoning"]].head()

Saved the 50 sampled reviews with llm_labels and llm_reasoning filled in: (50, 7)


,review_id,labels,llm_labels,llm_reasoning
1,a829e1ebbb4b6d7655b219b401c58d95,"['religion', 'age-appropriateness']","['age-appropriateness', 'religion', 'non-factual']","The reviewer complains that the age recommendation is inappropriate, criticizes the religious content for misrepresenting Christian views on science, and states that the book gives incorrect information."
7,e0337f5711a585a85a238d53f122ed69,['racism'],"['racism', 'bad-influence', 'age-appropriateness']",The reviewer complains that the book contains racist propaganda and expresses concern that it will be a bad influence on their young child who is too young to understand it.
21,621ca6c909f6fc3bd21639a2febed019,"['bad-influence', 'age-appropriateness', 'drugs/alcohol']","['age-appropriateness', 'bad-influence']","The reviewer complains that the book teaches children to steal and poach, making it a bad influence for kids."
24,692a015df0fac462a97fbe6d0d1edede,['age-appropriateness'],['age-appropriateness'],The reviewer explicitly states the book is inappropriate for a small child due to its textbook-like structure.
31,f620703a6bf924f1bc67ea825c90a81f,['scary/dark'],['scary/dark'],The reviewer specifically complains that the final story and the book in general were too dark.


#### 3. Accuracy and disagreements

With several labels per review there are a few ways to measure agreement, so we report them all:

- **Exact-match accuracy:** the share of reviews where Gemini's set of labels is *exactly* the hand-labeled set. This is the strictest number.
- **Per-label precision and recall:** for each label, `precision` = of the reviews Gemini gave this label, how many did the hand labels also have; `recall` = of the reviews with this hand label, how many did Gemini find.
  A *false positive* is a label Gemini added that the hand labels do not have; a *false negative* is a hand label that Gemini missed.
- **Micro precision / recall / F1:** the same idea counted over all labels together.
- **Average overlap (Jaccard):** how much the two label sets overlap on average (1 = identical, 0 = nothing in common).

In [26]:
def to_set(cell):
    try:
        return {str(x) for x in ast.literal_eval(cell)} if str(cell).strip() else set()
    except (ValueError, SyntaxError):
        return set()


def evaluate(table, results):
    """Compare the hand labels in `table` with Gemini's labels in `results`. Returns one row per review, one row per label, and a summary."""
    t = table[[ID_COL, TEXT_COL, HAND_COL, "notes"]].merge(
        results[["id", "llm_labels", "llm_reasoning"]], left_on=ID_COL, right_on="id", how="inner")
    t["hand_set"] = [hand_sets[r] for r in t[ID_COL]]
    t["llm_set"] = t["llm_labels"].apply(to_set)
    t["exact_match"] = [h == g for h, g in zip(t["hand_set"], t["llm_set"])]
    t["missed_by_llm"] = [", ".join(sorted(h - g)) for h, g in zip(t["hand_set"], t["llm_set"])]     # false negatives
    t["extra_from_llm"] = [", ".join(sorted(g - h)) for h, g in zip(t["hand_set"], t["llm_set"])]    # false positives
    t["jaccard"] = [1.0 if not (h | g) else len(h & g) / len(h | g) for h, g in zip(t["hand_set"], t["llm_set"])]

    rows = []
    for lab in LABELS + extra_hand_labels:      # labels Gemini cannot know are included, so their misses are visible
        h = np.array([lab in s for s in t["hand_set"]])
        g = np.array([lab in s for s in t["llm_set"]])
        tp, fp, fn = int((h & g).sum()), int((~h & g).sum()), int((h & ~g).sum())
        rows.append({
            "label": lab, "hand_count": int(h.sum()), "llm_count": int(g.sum()),
            "true_pos": tp, "false_pos": fp, "false_neg": fn,
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "recall": tp / (tp + fn) if tp + fn else np.nan,
        })
    per_label = pd.DataFrame(rows).set_index("label")

    tp, fp, fn = per_label["true_pos"].sum(), per_label["false_pos"].sum(), per_label["false_neg"].sum()
    p = tp / (tp + fp) if tp + fp else np.nan
    r = tp / (tp + fn) if tp + fn else np.nan
    summary = {
        "n_reviews": len(t),
        "exact_match_accuracy": t["exact_match"].mean(),
        "average_overlap_jaccard": t["jaccard"].mean(),
        "micro_precision": p,
        "micro_recall": r,
        "micro_f1": 2 * p * r / (p + r) if p + r else np.nan,
    }
    return t, per_label, summary

In [27]:
v1_eval, v1_per_label, v1_summary = evaluate(data, v1_results)

print(pd.Series(v1_summary).round(3).to_string())
v1_per_label.round(2)

n_reviews                  50.000
exact_match_accuracy        0.500
average_overlap_jaccard     0.753
micro_precision             0.811
micro_recall                0.874
micro_f1                    0.841


,hand_count,llm_count,true_pos,false_pos,false_neg,precision,recall
label,,,,,,,
age-appropriateness,37,37,34,3,3,0.92,0.92
religion,2,3,2,1,0,0.67,1.00
racism,9,10,9,1,0,0.90,1.00
body-shaming,0,0,0,0,0,NaN,NaN
bad-influence,11,13,6,7,5,0.46,0.55
scary/dark,13,12,12,0,1,1.00,0.92
illustrations,6,6,4,2,2,0.67,0.67
abuse,2,2,2,0,0,1.00,1.00
ableism,0,0,0,0,0,NaN,NaN


In [30]:
# The reviews where Gemini's labels are NOT exactly the hand labels. Read these as a group.
pd.set_option("display.max_colwidth", None)

v1_disagreements = v1_eval[~v1_eval["exact_match"]][
    [ID_COL, TEXT_COL, HAND_COL, "llm_labels", "missed_by_llm", "extra_from_llm", "llm_reasoning", "notes"]
]
v1_disagreements.to_csv(OUT_DIR / "v1_disagreements.csv", index=False)   # share this file with the group
print(len(v1_disagreements), "of", len(v1_eval), "reviews differ")
v1_disagreements.head(5)

25 of 50 reviews differ


,review_id,review_text,labels,llm_labels,missed_by_llm,extra_from_llm,llm_reasoning,notes
0,a829e1ebbb4b6d7655b219b401c58d95,"Once again, the author brings up issues to complex for the rhyming scheme. From pages, oops, sorry there are no page numbers. \n ""As he untied the string that held \n the book that he had hidden, \n he knew he could be punished, for \n his father had forbidden \n all books containing science. He \n and many people thought \n that if you were Christian, science \n should never be taught. \n For many living in the \n nineteenth century believed \n it challenged faith in God to question \n how things were conceived, \n like plants and animals, \n and even people too. \n But David longed to study and \n learn how all life grew."" \n Furthermore, her animosity between Christians and science misrepresents David Livingstone. In other reading, it appears to me that Mr. Livingstone's issue with science was not that it shouldn't be studied at all, but that Biblical study should take precedence in the education of his children. I would say this is also true of Christians today. It's not that science is scary, evil, or lacks value, but that the Bible should be preeminent. Meanwhile, young David Livingstone's issue was that his own scientific curiosity lead him to rebel against his father, which he knew God did not view favorably. \n For a superior treatment of Livingstone, see Sam Wellman's, David Livingstone, from the Young Readers Christian Library series: \n https://www.goodreads.com/review/show... \n This series is awful. Listed on CBD for ages 5-10, and on Amazon as 4 and up, this designation is completely off, even arbitrary. I would recommend these books, maybe, for preschool. For some reason, they decided to use rhyming verse, a literary style that greatly appeals to the preschool set. But in that format, the serious topics of missions, paganism, and the confrontation between light and darkness are addressed. \n This is everything I hate about cheery ""christian"" literature. A sing-song, happy-happy-clap-clap approach to the very real war of the ages is infuriatingly inappropriate in preparing our children to stand for the LORD. If this kind of God will never let anything uncomfortable happen to you sweet child theology is what you are looking for, Meloche's work will suit you fine. But if you are looking to raise up godly children who are equipped to know, love and serve the amazing, awesome, Creative God, do yourself a favor: Don't buy any books in this series. \n Excellent alternatives, which I have reviewed on Goodreads are: \n Christian Biographies for Young Readers (Series), Simonetta Carr \n https://www.goodreads.com/series/8395... \n Trial & Tribulation (A single book overview of Church History), Hannula \n https://www.goodreads.com/review/show... \n History Lives Series by Brandon & Mindy Withrow \n https://www.goodreads.com/review/show...","['religion', 'age-appropriateness']","['age-appropriateness', 'religion', 'non-factual']",,non-factual,"The reviewer complains that the age recommendation is inappropriate, criticizes the religious content for misrepresenting Christian views on science, and states that the book gives incorrect information.",copy
1,e0337f5711a585a85a238d53f122ed69,"What an infuriating book. I don't know what infuriates me more: that Kipling was a racist imperialist colonizer who believed firmly in white superiority and conveyed that in every word of these stories; or that Kipling is such a marvelous writer of the English language. \n Kipling the colonizer, imperialist, racist, supremicist, had no trouble at all mugging the oral traditions of the peoples his people colonized to tell his ""Just So Stories"" to his Best Beloved. No trouble at all mimicking their voices with disgusting condescension, rewriting origin tales, creating new origin tales, playfully interweaving the inevitability of England's rise as though fated (as he does so deftly in How the First Letter Was Written

**For each disagreement ask:** is Gemini wrong, are the hand labels wrong, or is the definition unclear?
Look first at labels with many false positives or false negatives in the table above, and at labels whose definitions overlap
(for example `scary/dark` and `violence`, or `bad-influence` and `age-appropriateness`).
The `notes` column is shown here only to help the group read the cases; Gemini never sees it.



#### 4. The group polishes the definitions (v2)

Edit the definitions below after the group meeting. Change the wording in `LABEL_RULES_V2`, add or remove lines in `EXTRA_RULES_V2`.
If you add a label, add it to `LABEL_RULES_V2`; the reply format is rebuilt from it automatically.